In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark.sql.functions import explode, arrays_zip, from_unixtime, col
from pyspark.sql.types import DateType

import os

In [2]:
from pyspark.sql import SparkSession

HADOOP_VERSION = "3.4.1"   # from step 1

spark = (
    SparkSession.builder.appName("FormatStock")
    .config("spark.jars.packages", f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

print(spark.conf.get("spark.jars.packages"))

org.apache.hadoop:hadoop-aws:3.4.1


In [3]:
spark

In [4]:
from dotenv import load_dotenv

load_dotenv()

path = os.environ["SPARK_APPLICATION_ARGS"]

In [5]:
path

'stock-market/NVDA'

```text
root
 |-- meta: struct
 |    |-- symbol: string
 |    |-- currency: string ...
 |-- timestamp: array<long>
 |-- indicators: struct
 |    |-- quote: array<struct>
 |         |-- open:   array<double>
 |         |-- high:   array<double>
 |         |-- low:    array<double>
 |         |-- close:  array<double>
 |         |-- volume: array<long>
```

In [6]:
df = spark.read.option("multiLine", "true").json(f"s3a://{path}/prices.json")
df.printSchema()

root
 |-- indicators: struct (nullable = true)
 |    |-- adjclose: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adjclose: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |-- quote: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- close: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- high: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- low: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- open: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- volume: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |-- meta: struct (nullable = true)
 |    |-- chartPreviousClose: double (nullab

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df.select("meta.symbol", "meta.longName", "meta.currency", "meta.exchangeName",
          "meta.regularMarketPrice", "meta.fiftyTwoWeekHigh", "meta.fiftyTwoWeekLow",
          "meta.range", "meta.dataGranularity").show(truncate=False)

+------+------------------+--------+------------+------------------+----------------+---------------+-----+---------------+
|symbol|longName          |currency|exchangeName|regularMarketPrice|fiftyTwoWeekHigh|fiftyTwoWeekLow|range|dataGranularity|
+------+------------------+--------+------------+------------------+----------------+---------------+-----+---------------+
|NVDA  |NVIDIA Corporation|USD     |NMS         |202.06            |243.37          |164.27         |2y   |1d             |
+------+------------------+--------+------------+------------------+----------------+---------------+-----+---------------+



In [8]:
df.select(
    F.size("timestamp").alias("n_timestamps"),
    F.size(F.col("indicators.quote")[0]["close"]).alias("n_closes"),
    F.size(F.col("indicators.adjclose")[0]["adjclose"]).alias("n_adjcloses"),
).show()

+------------+--------+-----------+
|n_timestamps|n_closes|n_adjcloses|
+------------+--------+-----------+
|         500|     500|        500|
+------------+--------+-----------+



In [9]:
df.select(
    F.col("meta.symbol").alias("symbol"),
    F.col("indicators.quote")[0].alias("q"),
    F.col("indicators.adjclose")[0]["adjclose"].alias("adj"),
    F.posexplode("timestamp").alias("i", "ts")
).select(
    "symbol",
    F.col("q.open")[F.col("i")].alias("Open"),
    F.col("q.close")[F.col('i')].alias("Close"),
    F.col("q.volume")[F.col("i")].alias("Volume"),
    F.col("adj")[F.col("i")].alias("adj_close"),
    F.to_date(F.from_unixtime("ts")).alias("Date")
).show(5, truncate=False)

+------+------------------+------------------+---------+------------------+----------+
|symbol|Open              |Close             |Volume   |adj_close         |Date      |
+------+------------------+------------------+---------+------------------+----------+
|NVDA  |137.2100067138672 |136.0500030517578 |187528200|135.6859130859375 |2024-11-04|
|NVDA  |137.4499969482422 |139.91000366210938|160537400|139.53558349609375|2024-11-05|
|NVDA  |142.9600067138672 |145.61000061035156|242043900|145.2203369140625 |2024-11-06|
|NVDA  |146.38999938964844|148.8800048828125 |207323300|148.48159790039062|2024-11-07|
|NVDA  |148.77000427246094|147.6300048828125 |175665800|147.2349395751953 |2024-11-08|
+------+------------------+------------------+---------+------------------+----------+
only showing top 5 rows


In [10]:
q = F.col("indicators.quote")[0]
adj = F.col("indicators.adjclose")[0]["adjclose"]

prices = (
    df.select(
        F.col("meta.symbol").alias("symbol"),
        F.posexplode("timestamp").alias("i", "ts"),
        q.alias("q"),
        adj.alias("adj"),
    )
    .select(
        "symbol",
        F.to_date(F.from_unixtime("ts")).alias("date"),
        F.col("q.open")[F.col("i")].alias("open"),
        F.col("q.high")[F.col("i")].alias("high"),
        F.col("q.low")[F.col("i")].alias("low"),
        F.col("q.close")[F.col("i")].alias("close"),
        F.col("adj")[F.col("i")].alias("adj_close"),
        F.col("q.volume")[F.col("i")].alias("volume"),
    )
)

prices.cache()          # keep it in memory: we'll use it many times below
prices.show(10)
prices.printSchema()

+------+----------+------------------+------------------+------------------+------------------+------------------+---------+
|symbol|      date|              open|              high|               low|             close|         adj_close|   volume|
+------+----------+------------------+------------------+------------------+------------------+------------------+---------+
|  NVDA|2024-11-04| 137.2100067138672| 138.9600067138672|135.57000732421875| 136.0500030517578| 135.6859130859375|187528200|
|  NVDA|2024-11-05| 137.4499969482422| 140.3699951171875| 137.3300018310547|139.91000366210938|139.53558349609375|160537400|
|  NVDA|2024-11-06| 142.9600067138672|146.49000549316406| 141.9600067138672|145.61000061035156| 145.2203369140625|242043900|
|  NVDA|2024-11-07|146.38999938964844|148.92999267578125| 146.1699981689453| 148.8800048828125|148.48159790039062|207323300|
|  NVDA|2024-11-08|148.77000427246094|149.77000427246094|146.25999450683594| 147.6300048828125| 147.2349395751953|175665800|


In [11]:
df.printSchema()

root
 |-- indicators: struct (nullable = true)
 |    |-- adjclose: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adjclose: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |-- quote: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- close: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- high: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- low: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- open: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- volume: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |-- meta: struct (nullable = true)
 |    |-- chartPreviousClose: double (nullab

In [12]:
df.select(
    F.col("meta.symbol").alias("symbol"),
    F.explode(F.col("indicators.quote")[0]["close"].alias("close")),
).show(truncate=False)

+------+------------------+
|symbol|col               |
+------+------------------+
|NVDA  |136.0500030517578 |
|NVDA  |139.91000366210938|
|NVDA  |145.61000061035156|
|NVDA  |148.8800048828125 |
|NVDA  |147.6300048828125 |
|NVDA  |145.25999450683594|
|NVDA  |148.2899932861328 |
|NVDA  |146.27000427246094|
|NVDA  |146.75999450683594|
|NVDA  |141.97999572753906|
|NVDA  |140.14999389648438|
|NVDA  |147.00999450683594|
|NVDA  |145.88999938964844|
|NVDA  |146.6699981689453 |
|NVDA  |141.9499969482422 |
|NVDA  |136.02000427246094|
|NVDA  |136.9199981689453 |
|NVDA  |135.33999633789062|
|NVDA  |138.25            |
|NVDA  |138.6300048828125 |
+------+------------------+
only showing top 20 rows


In [13]:
df.select("meta.symbol", "timestamp", F.col("indicators.quote")[0]["close"].alias("close")).show(4)

+------+--------------------+--------------------+
|symbol|           timestamp|               close|
+------+--------------------+--------------------+
|  NVDA|[1730730600, 1730...|[136.050003051757...|
+------+--------------------+--------------------+



In [14]:
tiny = spark.sql("SELECT 'NVDA' AS symbol, array(100, 105, 103) AS close")

tiny.select("symbol", F.explode("close")).show()
tiny.select("symbol", F.posexplode("close")).show()

+------+---+
|symbol|col|
+------+---+
|  NVDA|100|
|  NVDA|105|
|  NVDA|103|
+------+---+

+------+---+---+
|symbol|pos|col|
+------+---+---+
|  NVDA|  0|100|
|  NVDA|  1|105|
|  NVDA|  2|103|
+------+---+---+



In [15]:
df_exploded = df.select("timestamp", explode("indicators.quote").alias("quote")) \
                .select("timestamp", "quote.*")

df_exploded.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|           timestamp|               close|                high|                 low|                open|              volume|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|[1730730600, 1730...|[136.050003051757...|[138.960006713867...|[135.570007324218...|[137.210006713867...|[187528200, 16053...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+



`.*` means “take every field of this struct and make each one its own column”:

In [16]:
df.select("indicators.quote").show()

+--------------------+
|               quote|
+--------------------+
|[{[136.0500030517...|
+--------------------+



In [17]:
df.select(explode("indicators.quote")).show()

+--------------------+
|                 col|
+--------------------+
|{[136.05000305175...|
+--------------------+



In [18]:
df.select("timestamp", explode("indicators.quote")).show()

+--------------------+--------------------+
|           timestamp|                 col|
+--------------------+--------------------+
|[1730730600, 1730...|{[136.05000305175...|
+--------------------+--------------------+



In [19]:
df.select("timestamp", F.col("indicators.quote")[0].alias("quote"))

DataFrame[timestamp: array<bigint>, quote: struct<close:array<double>,high:array<double>,low:array<double>,open:array<double>,volume:array<bigint>>]

In [20]:
df.select("timestamp", F.col("indicators.quote")[0].alias("quote")).show()

+--------------------+--------------------+
|           timestamp|               quote|
+--------------------+--------------------+
|[1730730600, 1730...|{[136.05000305175...|
+--------------------+--------------------+



In [21]:
df.select("timestamp", F.col("indicators.quote")[0].alias("quote")).select("timestamp", "quote.*").show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|           timestamp|               close|                high|                 low|                open|              volume|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|[1730730600, 1730...|[136.050003051757...|[138.960006713867...|[135.570007324218...|[137.210006713867...|[187528200, 16053...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+



In [22]:
df_exploded = df.select(
    col("meta.symbol").alias("symbol"),
    "timestamp",
    explode("indicators.quote").alias("quote"),
    F.col("indicators.adjclose")[0]["adjclose"].alias("adjclose"),
).select("symbol", "timestamp", "adjclose", "quote.*")

df_exploded.show()

+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|symbol|           timestamp|            adjclose|               close|                high|                 low|                open|              volume|
+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|  NVDA|[1730730600, 1730...|[135.685913085937...|[136.050003051757...|[138.960006713867...|[135.570007324218...|[137.210006713867...|[187528200, 16053...|
+------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+



In [27]:
df_zipped = df_exploded.select(
    "symbol",
    arrays_zip("timestamp", "close", "high", "low", "open", "volume").alias("zipped")
)
df_zipped.show()

+------+--------------------+
|symbol|              zipped|
+------+--------------------+
|  NVDA|[{1730730600, 136...|
+------+--------------------+



```text
timestamp: [Mon,  Tue,  Wed ]
close:     [100,  105,  103 ]
high:      [101,  106,  104 ]
              │     │     │
              ▼     ▼     ▼
zipped:  [ {Mon,100,101,…}, {Tue,105,106,…}, {Wed,103,104,…} ]
```

In [29]:
daily = df_zipped.select("symbol", explode("zipped").alias("day")).select("symbol", "day.*")
daily.show(5)

+------+----------+------------------+------------------+------------------+------------------+---------+
|symbol| timestamp|             close|              high|               low|              open|   volume|
+------+----------+------------------+------------------+------------------+------------------+---------+
|  NVDA|1730730600| 136.0500030517578| 138.9600067138672|135.57000732421875| 137.2100067138672|187528200|
|  NVDA|1730817000|139.91000366210938| 140.3699951171875| 137.3300018310547| 137.4499969482422|160537400|
|  NVDA|1730903400|145.61000061035156|146.49000549316406| 141.9600067138672| 142.9600067138672|242043900|
|  NVDA|1730989800| 148.8800048828125|148.92999267578125| 146.1699981689453|146.38999938964844|207323300|
|  NVDA|1731076200| 147.6300048828125|149.77000427246094|146.25999450683594|148.77000427246094|175665800|
+------+----------+------------------+------------------+------------------+------------------+---------+
only showing top 5 rows


In [30]:
step = df_zipped.select(explode("zipped"))
step.printSchema()

root
 |-- col: struct (nullable = false)
 |    |-- timestamp: long (nullable = true)
 |    |-- close: double (nullable = true)
 |    |-- high: double (nullable = true)
 |    |-- low: double (nullable = true)
 |    |-- open: double (nullable = true)
 |    |-- volume: long (nullable = true)



In [31]:
daily = daily.withColumn("date", from_unixtime("timestamp").cast(DateType()))\
        .select("date", "symbol", "open", "low", "high", "close", "volume")
daily.show(5)

+----------+------+------------------+------------------+------------------+------------------+---------+
|      date|symbol|              open|               low|              high|             close|   volume|
+----------+------+------------------+------------------+------------------+------------------+---------+
|2024-11-04|  NVDA| 137.2100067138672|135.57000732421875| 138.9600067138672| 136.0500030517578|187528200|
|2024-11-05|  NVDA| 137.4499969482422| 137.3300018310547| 140.3699951171875|139.91000366210938|160537400|
|2024-11-06|  NVDA| 142.9600067138672| 141.9600067138672|146.49000549316406|145.61000061035156|242043900|
|2024-11-07|  NVDA|146.38999938964844| 146.1699981689453|148.92999267578125| 148.8800048828125|207323300|
|2024-11-08|  NVDA|148.77000427246094|146.25999450683594|149.77000427246094| 147.6300048828125|175665800|
+----------+------+------------------+------------------+------------------+------------------+---------+
only showing top 5 rows


Why the Spark job runs in its own container

1. Airflow’s container doesn’t have what Spark needs. A Spark job needs Java, Spark itself (version 3.3.0 to match your cluster), the hadoop-aws jars and PySpark. Airflow’s image has none of these. You could install them into Airflow, but that’s heavy and fragile.

2. It keeps dependencies apart. Airflow has hundreds of Python packages pinned to exact versions. Put Spark, PySpark and Java into the same environment and sooner or later versions clash, the same kind of conflict you hit with python-dotenv and uv add. In separate containers, each side has exactly what it needs.

3. Airflow should orchestrate, not do the heavy work. This is the most important idea, and an interview favourite. Airflow’s job is deciding when things run and in what order. If big data processing runs inside Airflow’s workers, it competes with the scheduler for memory and CPU, and one heavy job can slow down every other DAG. So Airflow just says “start this job”, and the work happens elsewhere.

4. The same image works everywhere. airflow/stock-app contains the script plus every dependency, fixed at specific versions. It runs the same on your laptop, a colleague’s or a server. You can also test it without Airflow at all: